# UAS Rancang Bangun Sistem AI

Kelompok:
1. Jose Garcia Puglisi (202304560019)
2. Novriani Ambarita (202304560038)
3. Citra Amalia Lestari (202304560046)
4. Alosia Wulandari Dyah Pramesti (202304560025)

# Algoritma yang Dipakai: YOLOv5

## Mount Drive untuk setor datasetnya, dan juga hasil model yang telah ditrain, serta hasil deteksinya.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%cd /content/drive/MyDrive/
%cd /content/drive/MyDrive/UAS_RancangAI_Pothole
!mkdir -p yolov5_UAS
%cd yolov5_UAS

/content/drive/MyDrive
/content/drive/MyDrive/UAS_RancangAI_Pothole
/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS


## Install semua dependencies/keperluan untuk model training.

In [3]:
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -r requirements.txt

fatal: destination path 'yolov5' already exists and is not an empty directory.
/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/yolov5


In [4]:
import os
os.environ["WANDB_DISABLED"] = "true"

!pip uninstall -y wandb

# Scenario 1: Data Drone Original (No Augments, No Added Data, No Extreme Imbalance)

In [ ]:
!mkdir -p /content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/original

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="XTSrqZ9VYpH5qaSRT3Vg")
project = rf.workspace("unlimitedbladeworks").project("pothole_detection_uas")
version = project.version(1)
dataset = version.download("yolov5")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Pothole_Detection_UAS-1 in yolov5pytorch:: 100%|██████████| 170/170 [00:01<00:00, 160.76it/s]


In [ ]:
print(dataset.location)
!ls {dataset.location}

/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/yolov5/Pothole_Detection_UAS-1
data.yaml  README.dataset.txt  README.roboflow.txt  test  train  valid


In [ ]:
BASE_DIR = "/content/drive/MyDrive/UAS_RancangAI_Pothole"
ALGO = "yolov5_UAS"
SCENARIO = "original"
EXP_NAME = "pothole_yolov5_original"

In [ ]:
!python train.py \
  --img 640 \
  --batch 16 \
  --epochs 100 \
  --data {dataset.location}/data.yaml \
  --weights yolov5s.pt \
  --project {BASE_DIR}/{ALGO}/{SCENARIO} \
  --name {EXP_NAME}

2026-01-01 08:33:31.696716: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767256411.716405    8296 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767256411.722457    8296 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767256411.738039    8296 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767256411.738063    8296 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767256411.738067    8296 computation_placer.cc:177] computation placer alr

In [ ]:
!sed -n '1,200p' {dataset.location}/data.yaml
!ls {dataset.location}/test/

names:
- crack
- pothole-KsFF
nc: 2
roboflow:
  license: CC BY 4.0
  project: pothole_detection_uas
  url: https://universe.roboflow.com/unlimitedbladeworks/pothole_detection_uas/dataset/1
  version: 1
  workspace: unlimitedbladeworks
test: ../test/images
train: Pothole_Detection_UAS-1/train/images
val: Pothole_Detection_UAS-1/valid/images
images	labels


In [ ]:
!python val.py \
  --img 640 \
  --batch 16 \
  --data {dataset.location}/data.yaml \
  --weights /content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/original/pothole_yolov5_original/weights/best.pt \
  --task test \
  --project {BASE_DIR}/{ALGO}/{SCENARIO} \
  --name pothole_yolov5_original_test

val: data=/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/yolov5/Pothole_Detection_UAS-1/data.yaml, weights=['/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/original/pothole_yolov5_original/weights/best.pt'], batch_size=16, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=test, device=, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/original, name=pothole_yolov5_original_test, exist_ok=False, half=False, dnn=False
YOLOv5 🚀 v7.0-453-geed9bc19 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)

Fusing layers... 
Model summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs
test: Scanning /content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/yolov5/Pothole_Detection_UAS-1/test/labels... 15 images, 0 backgrounds, 0 corrupt: 100% 15/15 [00:00<00:00, 110.95it/s]
test: New cache created: /content/dr

In [ ]:
!python detect.py \
  --weights /content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/original/pothole_yolov5_original/weights/best.pt \
  --img 640 \
  --source {dataset.location}/test/images \
  --project {BASE_DIR}/{ALGO}/{SCENARIO} \
  --name pothole_yolov5_original_detect

detect: weights=['/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/original/pothole_yolov5_original/weights/best.pt'], source=/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/yolov5/Pothole_Detection_UAS-1/test/images, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_format=0, save_csv=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/original, name=pothole_yolov5_original_detect, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-453-geed9bc19 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)

Fusing layers... 
Model summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs
image 1/15 /content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/yolov5/Pothole

# Scenario 2: Data Tambahan dari Google Maps

In [ ]:
!mkdir -p !mkdir -p /content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/added

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="XTSrqZ9VYpH5qaSRT3Vg")
project = rf.workspace("unlimitedbladeworks").project("pothole_detection_uas-2-maps")
version = project.version(1)
added_dataset = version.download("yolov5")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Pothole_Detection_UAS-2(Maps)-1 in yolov5pytorch:: 100%|██████████| 252/252 [00:01<00:00, 140.99it/s]


In [ ]:
print(added_dataset.location)

/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/yolov5/Pothole_Detection_UAS-2(Maps)-1
/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `ls /content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/yolov5/Pothole_Detection_UAS-2(Maps)-1'


In [ ]:
ADDED_DATASET = "/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/yolov5/Pothole_Detection_UAS-2-Maps-1"
SCENARIO = "added"
EXP_NAME = "pothole_yolov5_added"

!python train.py \
  --img 640 \
  --batch 16 \
  --epochs 100 \
  --data {ADDED_DATASET}/data.yaml \
  --weights yolov5s.pt \
  --project {BASE_DIR}/{ALGO}/{SCENARIO} \
  --name {EXP_NAME}

2026-01-01 09:08:29.077748: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767258509.111858   17440 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767258509.122004   17440 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767258509.146558   17440 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767258509.146592   17440 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767258509.146601   17440 computation_placer.cc:177] computation placer alr

In [ ]:
!python val.py \
  --img 640 \
  --batch 16 \
  --data {ADDED_DATASET}/data.yaml \
  --weights /content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/added/pothole_yolov5_added/weights/best.pt \
  --task test \
  --project {BASE_DIR}/{ALGO}/{SCENARIO} \
  --name pothole_yolov5_added_test

val: data=/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/yolov5/Pothole_Detection_UAS-2-Maps-1/data.yaml, weights=['/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/added/pothole_yolov5_added/weights/best.pt'], batch_size=16, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=test, device=, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/added, name=pothole_yolov5_added_test, exist_ok=False, half=False, dnn=False
YOLOv5 🚀 v7.0-453-geed9bc19 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)

Fusing layers... 
Model summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs
test: Scanning /content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/yolov5/Pothole_Detection_UAS-2-Maps-1/test/labels... 20 images, 0 backgrounds, 0 corrupt: 100% 20/20 [00:00<00:00, 98.69it/s] 
test: New cache created: /content/

In [ ]:
!python detect.py \
  --weights /content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/added/pothole_yolov5_added/weights/best.pt \
  --img 640 \
  --source {ADDED_DATASET}/test/images \
  --project {BASE_DIR}/{ALGO}/{SCENARIO} \
  --name pothole_yolov5_added_detect

detect: weights=['/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/added/pothole_yolov5_added/weights/best.pt'], source=/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/yolov5/Pothole_Detection_UAS-2-Maps-1/test/images, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_format=0, save_csv=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/added, name=pothole_yolov5_added_detect, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-453-geed9bc19 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)

Fusing layers... 
Model summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs
image 1/20 /content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/yolov5/Pothole_Dete

# Scenario 3: Imbalanced Dataset

In [ ]:
!mkdir -p !mkdir -p /content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/imbalanced

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="XTSrqZ9VYpH5qaSRT3Vg")
project = rf.workspace("unlimitedbladeworks").project("pothole_detection_uas-3-imbalanced")
version = project.version(1)
imbalanced_dataset = version.download("yolov5")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Pothole_Detection_UAS-3(Imbalanced)-1 in yolov5pytorch:: 100%|██████████| 128/128 [00:00<00:00, 162.84it/s]


In [ ]:
IMBALANCED_DATASET = "/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/yolov5/Pothole_Detection_UAS-3-Imbalanced-1"
SCENARIO = "imbalanced"
EXP_NAME = "pothole_yolov5_imbalanced"

!python train.py \
  --img 640 \
  --batch 16 \
  --epochs 100 \
  --data {IMBALANCED_DATASET}/data.yaml \
  --weights yolov5s.pt \
  --project {BASE_DIR}/{ALGO}/{SCENARIO} \
  --name {EXP_NAME}

2026-01-01 09:20:02.301371: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767259202.321109   20554 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767259202.327396   20554 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767259202.342587   20554 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767259202.342614   20554 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767259202.342618   20554 computation_placer.cc:177] computation placer alr

In [ ]:
!python val.py \
  --img 640 \
  --batch 16 \
  --data {IMBALANCED_DATASET}/data.yaml \
  --weights /content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/imbalanced/pothole_yolov5_imbalanced/weights/best.pt \
  --task test \
  --project {BASE_DIR}/{ALGO}/{SCENARIO} \
  --name pothole_yolov5_imbalanced_test

val: data=/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/yolov5/Pothole_Detection_UAS-3-Imbalanced-1/data.yaml, weights=['/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/imbalanced/pothole_yolov5_imbalanced/weights/best.pt'], batch_size=16, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=test, device=, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/imbalanced, name=pothole_yolov5_imbalanced_test, exist_ok=False, half=False, dnn=False
YOLOv5 🚀 v7.0-453-geed9bc19 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)

Fusing layers... 
Model summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs
test: Scanning /content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/yolov5/Pothole_Detection_UAS-3-Imbalanced-1/test/labels... 10 images, 0 backgrounds, 0 corrupt: 100% 10/10 [00:00<00:00, 122.37it/s]
te

In [ ]:
!python detect.py \
  --weights /content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/imbalanced/pothole_yolov5_imbalanced/weights/best.pt \
  --img 640 \
  --source {IMBALANCED_DATASET}/test/images \
  --project {BASE_DIR}/{ALGO}/{SCENARIO} \
  --name pothole_yolov5_imbalanced_detect

detect: weights=['/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/imbalanced/pothole_yolov5_imbalanced/weights/best.pt'], source=/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/yolov5/Pothole_Detection_UAS-3-Imbalanced-1/test/images, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_format=0, save_csv=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/imbalanced, name=pothole_yolov5_imbalanced_detect, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-453-geed9bc19 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)

Fusing layers... 
Model summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs
image 1/10 /content/drive/MyDrive/UAS_RancangAI_Pothole/yolo

# Scenario 4: Augmented Dataset

In [5]:
!mkdir -p /content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/augmented

In [6]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="XTSrqZ9VYpH5qaSRT3Vg")
project = rf.workspace("unlimitedbladeworks").project("pothole_detection_uas-4-augment")
version = project.version(2)
augmented_dataset = version.download("yolov5")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Pothole_Detection_UAS-4(Augment)-2 in yolov5pytorch:: 100%|██████████| 366/366 [00:02<00:00, 150.37it/s]


In [26]:
BASE_DIR = "/content/drive/MyDrive/UAS_RancangAI_Pothole"
ALGO = "yolov5_UAS"
SCENARIO = "augmented"
EXP_NAME = "pothole_yolov5_augmented"

!python train.py \
  --img 640 \
  --batch 16 \
  --epochs 100 \
  --data "/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/yolov5/Pothole_Detection_UAS-4-Augment-2/data.yaml" \
  --weights yolov5s.pt \
  --project {BASE_DIR}/{ALGO}/{SCENARIO} \
  --name {EXP_NAME}

2026-01-03 13:35:18.203351: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767447318.222385   10088 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767447318.228321   10088 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767447318.244116   10088 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767447318.244144   10088 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767447318.244149   10088 computation_placer.cc:177] computation placer alr

In [27]:
!python val.py \
  --img 640 \
  --batch 16 \
  --data "/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/yolov5/Pothole_Detection_UAS-4-Augment-2/data.yaml" \
  --weights /content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/augmented/pothole_yolov5_augmented/weights/best.pt \
  --task test \
  --project {BASE_DIR}/{ALGO}/{SCENARIO} \
  --name pothole_yolov5_augmented_test

val: data=/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/yolov5/Pothole_Detection_UAS-4-Augment-2/data.yaml, weights=['/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/augmented/pothole_yolov5_augmented/weights/best.pt'], batch_size=16, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=test, device=, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/augmented, name=pothole_yolov5_augmented_test, exist_ok=False, half=False, dnn=False
YOLOv5 🚀 v7.0-453-geed9bc19 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)

Fusing layers... 
Model summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs
test: Scanning /content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/yolov5/Pothole_Detection_UAS-4-Augment-2/test/labels... 15 images, 0 backgrounds, 0 corrupt: 100% 15/15 [00:00<00:00, 164.81it/s]
test: New ca

In [28]:
!python detect.py \
  --weights /content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/augmented/pothole_yolov5_augmented/weights/best.pt \
  --img 640 \
  --source "/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/yolov5/Pothole_Detection_UAS-4-Augment-2/test/images" \
  --project {BASE_DIR}/{ALGO}/{SCENARIO} \
  --name pothole_yolov5_augmented_detect

detect: weights=['/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/augmented/pothole_yolov5_augmented/weights/best.pt'], source=/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/yolov5/Pothole_Detection_UAS-4-Augment-2/test/images, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_format=0, save_csv=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=/content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/augmented, name=pothole_yolov5_augmented_detect, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-453-geed9bc19 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)

Fusing layers... 
Model summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs
image 1/15 /content/drive/MyDrive/UAS_RancangAI_Pothole/yolov5_UAS/